Для Анечки:
КАЧАЕМ ПИТОН 3.11!!
https://www.python.org/downloads/release/python-3119/

**вместо этого или рядом запиши свой пусть до папики с venv**

cd "C:\Users\Mihail\Desktop\ДЗ\5сем\Курсач"

**это если при активации venv вылезла ошибка**

Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process

**Это для активации venv, чтобы он изолировал и всё работало как надо**

.\venv\Scripts\Activate.ps1


**В строку пишем, если нужна проверка**

pip list -- список библотек

### Команды git (в командной строке снизу). Выполнять поочерёдно
git status (смотрим количество обновляемых файлов, какие именно, в какой ветке редактирования находимся.)

перед началом работы, если файлы есть у тебя на компе, то

git status

git pull (подгружает последнюю версию ветки)

**Если сделаешь коммит в main, то яйца пришью и оторву**

потом, ЕСЛИ ОБНОВЛЯЛА ФАЙЛЫ И ХОЧЕШЬ ОСТАВИТЬ ЭТО, ИНАЧЕ НЕ НАДО

git add .

git status кол-во, ветка не main

git push




# Смотрим

In [7]:
pip install -r requirements.txt

^C
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import math
from datetime import datetime
import os

In [ ]:
# Создаем папку для результатов, если её нет
os.makedirs('results', exist_ok=True)

# 1. ЗАГРУЗКА ДАННЫХ
file_path = 'data/transport_2025-04-21.csv'
df = pd.read_csv(file_path)

display(df.head(3))

,id,api_id,created_at,lasttime,lon,lat,gos_num,rid,rnum,rtype,low_floor,dir_api
0,2897,1861785000096408,21.04.2025 03:02:52,21.04.2025 00:03:12,47.475026,55.870801,В343КУ/21,398,173,М,0,353
1,2898,1861785000096408,21.04.2025 03:03:52,21.04.2025 00:04:13,47.475026,55.870801,В343КУ/21,398,173,М,0,353
2,2899,1861785000096408,21.04.2025 03:04:53,21.04.2025 00:05:13,47.475026,55.870801,В343КУ/21,398,173,М,0,353


In [32]:
# Путь к папке с данными (относительно текущей папки проекта)
folder_path = 'data'

# Получаем список всех файлов и папок внутри
csv_files = os.listdir(folder_path)

print(f"📁 В папке '{folder_path}' найдено {len(csv_files)} элементов")
print(csv_files)

📁 В папке 'data' найдено 70 элементов
['transport_2025-04-21.csv', 'transport_2025-04-22.csv', 'transport_2025-04-23.csv', 'transport_2025-04-24.csv', 'transport_2025-04-25.csv', 'transport_2025-04-26.csv', 'transport_2025-04-27.csv', 'transport_2025-04-28.csv', 'transport_2025-05-04.csv', 'transport_2025-05-05.csv', 'transport_2025-05-06.csv', 'transport_2025-05-07.csv', 'transport_2025-05-08.csv', 'transport_2025-05-09.csv', 'transport_2025-05-10.csv', 'transport_2025-05-11.csv', 'transport_2025-05-12.csv', 'transport_2025-05-13.csv', 'transport_2025-05-14.csv', 'transport_2025-05-15.csv', 'transport_2025-05-16.csv', 'transport_2025-05-17.csv', 'transport_2025-05-18.csv', 'transport_2025-05-19.csv', 'transport_2025-05-20.csv', 'transport_2025-05-21.csv', 'transport_2025-05-22.csv', 'transport_2025-05-23.csv', 'transport_2025-05-24.csv', 'transport_2025-05-25.csv', 'transport_2025-05-26.csv', 'transport_2025-05-27.csv', 'transport_2025-05-28.csv', 'transport_2025-05-29.csv', 'transpor

### Функция преобразования таблицы в базовый рабочий вид

In [ ]:
def process_transport_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Очищает сырые данные GPS-треков:
    1. Удаляет лишние технические колонки
    2. Преобразует строку времени в объект datetime
    3. Переименовываем rtype (тип маршрута междугородний, городской)
    4. Создаем примерные метки остановок
    """
    # Делаем копию, чтобы не менять исходный DataFrame
    df_clean = df.copy()
    
    # 1. Удаляем ненужные колонки
    columns_to_drop = ['id', 'api_id', 'created_at', 'low_floor']
    df_clean = df_clean.drop(columns=[col for col in columns_to_drop if col in df_clean.columns], errors='ignore')
    
    # 2. Преобразуем время в формат datetime
    df_clean['lasttime_dt'] = pd.to_datetime(df_clean['lasttime'], format='%d.%m.%Y %H:%M:%S')
    df_clean.drop('lasttime', axis = 1, inplace = True)

    # 3. Переименовываем rtype
    # (например, автобус, троллейбус, маршрутное такси.)

    # 4. Считаем примерные координаты каждой из остановок (точность около 50м) чтобы посчитать кол-во остановок в df
    df['lat_rounded'] = df['lat'].round(4)
    df['lon_rounded'] = df['lon'].round(4)
    return df_clean


process_transport_data(df).head(3)

,lon,lat,gos_num,rid,rnum,rtype,dir_api,lat_rounded,lon_rounded,lasttime_dt
0,47.475026,55.870801,В343КУ/21,398,173,М,353,55.8708,47.475,2025-04-21 00:03:12
1,47.475026,55.870801,В343КУ/21,398,173,М,353,55.8708,47.475,2025-04-21 00:04:13
2,47.475026,55.870801,В343КУ/21,398,173,М,353,55.8708,47.475,2025-04-21 00:05:13


In [54]:
display(df[df['rtype'] == 'М'].head(10))

,id,api_id,created_at,lasttime,lon,lat,gos_num,rid,rnum,rtype,low_floor,dir_api,lat_rounded,lon_rounded
0,2897,1861785000096408,21.04.2025 03:02:52,21.04.2025 00:03:12,47.475026,55.870801,В343КУ/21,398,173,М,0,353,55.8708,47.475
1,2898,1861785000096408,21.04.2025 03:03:52,21.04.2025 00:04:13,47.475026,55.870801,В343КУ/21,398,173,М,0,353,55.8708,47.475
2,2899,1861785000096408,21.04.2025 03:04:53,21.04.2025 00:05:13,47.475026,55.870801,В343КУ/21,398,173,М,0,353,55.8708,47.475
3,2900,1861785000096408,21.04.2025 03:05:53,21.04.2025 00:06:13,47.475026,55.870801,В343КУ/21,398,173,М,0,353,55.8708,47.475
4,2901,1861785000096408,21.04.2025 03:06:53,21.04.2025 00:07:13,47.475026,55.870801,В343КУ/21,398,173,М,0,353,55.8708,47.475
5,2902,1861785000096408,21.04.2025 03:07:53,21.04.2025 00:08:13,47.475026,55.870801,В343КУ/21,398,173,М,0,353,55.8708,47.475
6,2903,1861785000096408,21.04.2025 03:08:53,21.04.2025 00:09:13,47.475026,55.870801,В343КУ/21,398,173,М,0,353,55.8708,47.475
7,2904,1861785000096408,21.04.2025 03:09:53,21.04.2025 00:10:13,47.475026,55.870801,В343КУ/21,398,173,М,0,353,55.8708,47.475
8,2905,1861785000096408,21.04.2025 03:10:54,21.04.2025 00:11:13,47.475026,55.870801,В343КУ/21,398,173,М,0,353,55.8708,47.475
9,2906,1861785000096408,21.04.2025 03:11:54,21.04.2025 00:12:14,47.475026,55.870801,В343КУ/21,398,173,М,0,353,55.8708,47.475


In [ ]:
def create_big_csv(list_of_csv: list) -> pd.DataFrame:
    """
    Создаёт одну большую таблицу из списка CSV-файлов.
    Проверяет совпадение атрибутов (колонок) во всех файлах.
    """
    dfs = []
    reference_columns = None
    
    for file in list_of_csv:
        file_path = 'data/' + file
        
        # Читаем и обрабатываем файл
        df = pd.read_csv(file_path)
        df = process_transport_data(df)
        
        # Получаем текущий набор колонок как множество (set) для быстрого сравнения
        current_columns = set(df.columns)
        
        # Логика проверки
        if reference_columns is None:
            # Первый файл задаёт эталон
            reference_columns = current_columns
            print(f"📋 Эталонный набор колонок (по файлу {file}):")
            print(f"   {sorted(list(reference_columns))}\n")
        else:
            # Сравниваем текущий файл с эталоном
            if current_columns != reference_columns:
                missing_cols = reference_columns - current_columns
                extra_cols = current_columns - reference_columns
                
                error_msg = f"❌ Ошибка структуры в файле '{file}'!\n"
                if missing_cols:
                    error_msg += f"   - Отсутствуют колонки: {missing_cols}\n"
                if extra_cols:
                    error_msg += f"   - Лишние колонки: {extra_cols}\n"
                
                raise ValueError(error_msg)
                
        dfs.append(df)

    # Если всё хорошо, склеиваем таблицы
    big_df = pd.concat(dfs, ignore_index=True)
    
    # Добавляем сквозной уникальный id
    big_df.insert(0, 'id', range(1, len(big_df) + 1))
    
    print(f"✅ Успешно объединено {len(list_of_csv)} файлов.")
    print(f"📊 Итоговый размер: {len(big_df)} строк, {len(big_df.columns)} колонок.")
    
    return big_df

In [43]:
unique_counts = df.nunique().to_frame(name='Уникальных значений')
display(unique_counts)

,Уникальных значений
id,371772
api_id,532
created_at,1221
lasttime,60569
lon,122786
lat,55580
gos_num,531
rid,102
rnum,51
rtype,3


In [45]:
cols_to_analyze_test = [
    'gos_num',        # Автобусы
    'rnum',           # Номера маршрутов
    'rid',            # ID маршрутов
    'dir_api',        # Направления
    'lat_rounded',    # Округленная широта (для поиска кластеров/депо)
    'lon_rounded'     # Округленная долгота
]

def analyze(df: pd.DataFrame, cols_to_analyze):
    for col in cols_to_analyze:
        print(f"\n Атрибут: '{col}' (Всего уникальных: {df[col].nunique()})")
        
        # value_counts() считает повторения и сразу сортирует по убыванию
        top_values = df[col].value_counts().head(10)
        
        # Форматируем вывод в красивую таблицу
        result_df = top_values.to_frame(name='Количество повторений')
        result_df.index.name = 'Уникальное значение'

        return display(result_df)

analyze(df, cols_to_analyze_test)
sum_df = create_big_csv(csv_files)
analyze(sum_df, cols_to_analyze)


 Атрибут: 'gos_num' (Всего уникальных: 531)


,Количество повторений
Уникальное значение,
М125КС (35),2701
926,1498
921,1498
1180,1495
965,1494
1142,1469
939,1464
952,1464
1256,1459


📋 Эталонный набор колонок (по файлу transport_2025-04-21.csv):
   ['dir_api', 'gos_num', 'lasttime_dt', 'lat', 'lon', 'rid', 'rnum', 'rtype']



C:\Users\Mihail\AppData\Local\Temp\ipykernel_20012\4143345227.py:13: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
C:\Users\Mihail\AppData\Local\Temp\ipykernel_20012\4143345227.py:13: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
C:\Users\Mihail\AppData\Local\Temp\ipykernel_20012\4143345227.py:13: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


✅ Успешно объединено 70 файлов.
📊 Итоговый размер: 31437271 строк, 9 колонок.

 Атрибут: 'gos_num' (Всего уникальных: 785)


,Количество повторений
Уникальное значение,
949,86221
1238,85690
945,84104
1239,83825
925,83422
1186,83160
950,82542
940,82337
1195,81507


In [47]:
# Путь, куда будем сохранять (в папку data)
output_path = 'results/sum_transport_data.csv'

# Сохраняем DataFrame в CSV
sum_df.to_csv(
    output_path, 
    index=False,                # Не сохраняем встроенный индекс pandas (у нас уже есть свой 'id')
    encoding='utf-8-sig',       # Критически важно! Позволяет Excel корректно открывать кириллицу
    date_format='%Y-%m-%d %H:%M:%S' # Сохраняем datetime в понятном строковом формате
)

print(f"✅ Файл успешно сохранен: {output_path}")
print(f" Размер файла: {os.path.getsize(output_path) / (1024*1024):.2f} МБ")

✅ Файл успешно сохранен: results/sum_transport_data.csv
 Размер файла: 2269.21 МБ


### Создаём список, в котором есть все названия таблиц

In [ ]:
# ==========================================================
# 2. ТАБЛИЦА: Автобусы и их маршруты
# ==========================================================
# Группируем по гос. номеру автобуса и смотрим, на каких маршрутах он работал
# 1. Получаем список всех CSV-файлов в папке data
folder_path = 'data'
csv_files = [f for f in os.listdir(folder_path)]

print(f" Найдено {len(csv_files)} файлов для обработки")

# 2. Читаем каждый файл и складываем в список
dfs = []
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    temp_df = pd.read_csv(file_path)

    # Добавляем колонку с именем файла (датой), чтобы потом можно было анализировать по дням
    temp_df['file_date'] = file.replace('transport_', '').replace('.csv', '')
    
    dfs.append(temp_df)
    print(f" Загружен: {file} ({len(temp_df)} записей)")

# 3. Объединяем все файлы в один большой DataFrame
df = pd.concat(dfs, ignore_index=True)
print(f"\n📊 Итого в объединённой таблице: {len(df)} записей")

# ==========================================================
# 4. ТАБЛИЦА: Автобусы и их маршруты (по всем дням сразу)
# ==========================================================
bus_routes = df.groupby('gos_num').agg(
    routes_list=('rnum', lambda x: list(x.astype(str).unique())),
    route_ids=('rid', lambda x: list(x.unique())),
    directions=('dir_api', lambda x: list(x.unique())),
    total_pings=('id', 'count'),
    days_active=('file_date', lambda x: len(x.unique()))  # Сколько разных дней автобус работал
).reset_index()

# Сортируем по активности
bus_routes = bus_routes.sort_values(by='total_pings', ascending=False)

# Сохраняем результат
bus_routes.to_csv('results/buses_and_routes.csv', index=False, encoding='utf-8-sig')
print("\nТаблица 'Автобусы и маршруты' сохранена в results/buses_and_routes.csv")
display(bus_routes.head())


✅ Таблица 'Автобусы и маршруты' сохранена в results/buses_and_routes.csv


,gos_num,routes_list,route_ids,directions,total_pings
427,М125КС (35),[35],"[98, 99]","[436, 76, 433, 72, 64, 58, 427, 67, 68, 407, 3...",2701
122,921,[15],"[343, 26]","[242, 283, 292, 291, 307, 305, 267, 309, 239, ...",1498
126,926,[20],"[35, 36]","[335, 205, 168, 169, 170, 166, 436, 84, 75, 43...",1498
21,1180,[62],"[406, 407]","[280, 296, 274, 287, 278, 265, 255, 164, 181, ...",1495
158,965,[14],"[371, 23]","[168, 170, 389, 32, 29, 52, 140, 150, 162, 439...",1494


In [30]:
print(bus_routes['gos_num'].unique())

len(bus_routes['gos_num'].unique())

['М125КС (35)' '921' '926' '1180' '965' '1142' '952' '939' '1256' '1233'
 '981' '832' '993' '999' '954' '1182' '928' '865' 'М732КТ' '1231' '1260'
 '995' '991' '964' '949' '881' 'М117КС (35)' '1186' 'Т531РР/777' '842'
 '950' '869' '1222' '1237' '1238' '963' '955' '938' '1252' '1209' '922'
 '1201' '997' '936' '941' 'О516ОХ/777' '802' 'Н491КВ/21' '1239' '1234'
 '976' 'М142КС (35)' 'О889ОХ/777' 'АР137 (15)' '1278' '1199' 'М873РК (41)'
 '974' 'АР164 (45)' 'АР167 (45)' 'К033НР (43)' '1279' 'М813ХР/21'
 'М023КС (35)' '911' 'М016КС (35)' '946' 'С306РА/777' 'АР103 (46)'
 'В158РТ/21' '817' '966' 'М231РК (41)' 'АР136 (15)' 'М164КС (35)'
 'К617КН (43)' '1203' 'К184КР (54)' 'АО882  (47)' 'К143МО (33)'
 'К931КХ  (47)' 'АР190 (26)' 'АР175 (45)' 'О627ТС/777' 'М194КС (35)'
 'М157РА (41)' 'АР174 (15)' 'О720ТС/777' 'К720РА  (47)' '885'
 'М168КС (35)' '977' 'С923РА/777' 'К382КХ (47)' 'К513УР (42)'
 'АО884  (47)' 'К261ХМ/21' 'К183КН (43)' 'К810МУ (54)' 'К770КН (54)'
 'АР173 (45)' 'АР163 (45)' 'М325КС (35)'

531